# Funnel Analysis & Drop-Off Detection

This notebook analyzes sequential user conversion through a 6-stage signup funnel:
1. **Tracking Volume Across Sequential Stages** (`Sign Up` to `First Purchase`).
2. **Calculating Drop-Off and Completion Rates** between consecutive stages.
3. **Visualizing Funnel Progress** using annotated bar charts.
4. **Quantifying Financial Impact** of lost potential customers ($100 LTV per user).
5. **Formulating Actionable Root Cause Hypotheses and A/B Test Strategy**.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

filepath = '../data/raw/user_signup_funnel.csv'
if not os.path.exists(filepath):
    filepath = 'data/raw/user_signup_funnel.csv'

df = pd.read_csv(filepath)
print(f"Loaded signup funnel dataset containing {len(df):,} user records.")

## Task 1: Define Funnel Stages and Count User Volume

In [2]:
stages = {
    'Sign Up': len(df[df['signup_completed'] == 1]),
    'Email Entered': len(df[df['email_entered'] == 1]),
    'Password Created': len(df[df['password_created'] == 1]),
    'Email Verified': len(df[df['email_verified'] == 1]),
    'Payment Added': len(df[df['payment_added'] == 1]),
    'First Purchase': len(df[df['first_purchase'] == 1])
}

for stage_name, count in stages.items():
    print(f"{stage_name:<18}: {count:,} users")

## Task 2: Compute Drop-Off Rate & Completion Rates Between Stages

In [3]:
stage_list = list(stages.values())
stage_names = list(stages.keys())
drop_off = []

for i in range(len(stage_list) - 1):
    users_before = stage_list[i]
    users_after = stage_list[i + 1]
    users_lost = users_before - users_after
    drop_pct = (users_lost / users_before) * 100
    comp_pct = (users_after / users_before) * 100
    
    drop_off.append({
        'from_stage': stage_names[i],
        'to_stage': stage_names[i + 1],
        'users_before': users_before,
        'users_after': users_after,
        'users_lost': users_lost,
        'completion_rate': f"{comp_pct:.1f}%",
        'drop_rate': f"{drop_pct:.1f}%",
        'drop_pct_num': drop_pct
    })

funnel_df = pd.DataFrame(drop_off)
print(funnel_df[['from_stage', 'to_stage', 'users_lost', 'completion_rate', 'drop_rate']])

## Task 3: Visualize Funnel Bar Chart

In [4]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#3b82f6', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6', '#ec4899']
bars = ax.bar(stages.keys(), stages.values(), color=colors, edgecolor='black', linewidth=0.8)

ax.set_ylabel('Users Volume', fontsize=12, fontweight='bold')
ax.set_xlabel('Funnel Stage', fontsize=12, fontweight='bold')
ax.set_title('User Conversion Funnel: Volume by Sequential Stage', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(stages.values()) * 1.15)
ax.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2.0, height + 150, f"{int(height):,}", ha='center', va='bottom', fontweight='bold')

plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Task 4: Financial Revenue Impact & Priority Ranking

In [5]:
revenue_per_customer = 100
funnel_df['revenue_lost'] = funnel_df['users_lost'] * revenue_per_customer
funnel_df['priority'] = funnel_df['revenue_lost'].apply(lambda x: 'HIGH' if x >= 100000 else 'MEDIUM')

ranked_df = funnel_df.sort_values('drop_pct_num', ascending=False)
print('Funnel Bottlenecks Ranked by Drop Rate %:')
print(ranked_df[['from_stage', 'to_stage', 'users_lost', 'drop_rate', 'revenue_lost', 'priority']])

## Task 5: Strategic Bottleneck Recommendations & Projected Impact

In [6]:
highest_impact = funnel_df.loc[funnel_df['drop_pct_num'].idxmax()]
recoverable_users = int(highest_impact['users_lost'] * 0.1)
recoverable_revenue = recoverable_users * revenue_per_customer

print(f"Critical Bottleneck: {highest_impact['from_stage']} -> {highest_impact['to_stage']}")
print(f"Drop Rate          : {highest_impact['drop_rate']} ({highest_impact['users_lost']:,} users lost)")
print(f"10% Fix Recovery   : +{recoverable_users:,} users (+${recoverable_revenue:,.0f} revenue)")